# Notebook 13 — Tox21: ECFP4 + XGBoost Baseline with SHAP

This notebook evaluates the ECFP4 + XGBoost fingerprint baseline on all 12 Tox21 assays
and provides chemically interpretable explanations via SHAP.

**Why fingerprints?**  
Each ECFP bit encodes a specific circular atom environment (substructure).  
SHAP values on fingerprint bits can be decoded back to those substructures,
giving direct, faithful, atom-level explanations — no post-hoc attribution
approximation, no compensation between modalities.

---

## Sections
1. Setup
2. Load models + test-set artefacts
3. Performance — per-task AUC table vs. SMILESGNN
4. Global SHAP — which substructures drive each assay?
5. Molecule-level SHAP — atom-highlighted visualizations
6. Top toxic substructure gallery (decode high-SHAP bits)


## 1. Setup

In [ ]:
import sys, pickle, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import shap
from pathlib import Path
from rdkit import Chem
from rdkit.Chem import AllChem, Draw
from rdkit.Chem.Draw import SimilarityMaps
from IPython.display import display

from src.datasets import get_task_config
from src.fingerprint import smiles_to_ecfp, shap_bits_to_atom_weights

MODEL_DIR  = Path('../models/tox21_fingerprint_model')
SMGNN_DIR  = Path('../models/tox21_smilesgnn_model')
RADIUS, NBITS = 2, 2048

task_config = get_task_config('tox21')
TASKS = task_config.task_names
print(f'Tasks: {TASKS}')

## 2. Load models + test-set artefacts

In [ ]:
# Load XGBoost models and SHAP explainers
models, explainers = {}, {}
for task in TASKS:
    mp = MODEL_DIR / 'models' / f'{task}.pkl'
    ep = MODEL_DIR / 'shap'   / f'{task}.pkl'
    if mp.exists():
        with open(mp, 'rb') as f: models[task]    = pickle.load(f)
        with open(ep, 'rb') as f: explainers[task] = pickle.load(f)

print(f'Loaded {len(models)} models')

# Load test-set artefacts
X_test     = np.load(MODEL_DIR / 'X_test.npy')
y_test     = np.load(MODEL_DIR / 'test_labels.npy')   # (N, 12), NaN for missing
with open(MODEL_DIR / 'test_smiles.pkl', 'rb') as f:
    test_smiles = pickle.load(f)

print(f'Test set  : {X_test.shape[0]} compounds, {X_test.shape[1]} bits')

## 3. Performance — per-task AUC vs. SMILESGNN

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score

# SMILESGNN results (from previous run)
smgnn_metrics_path = SMGNN_DIR / 'tox21_smilesgnn_metrics.txt'
smgnn_auc = {}
if smgnn_metrics_path.exists():
    with open(smgnn_metrics_path) as f:
        for line in f:
            if line.startswith('test_auc_'):
                k, v = line.strip().split(': ')
                task = k.replace('test_auc_', '')
                smgnn_auc[task] = float(v)

# Compute fingerprint model AUC per task
rows = []
for t, task in enumerate(TASKS):
    if task not in models:
        continue
    mask  = ~np.isnan(y_test[:, t])
    y_true = y_test[mask, t]
    probs  = models[task].predict_proba(X_test[mask])[:, 1]
    auc    = roc_auc_score(y_true, probs) if len(np.unique(y_true)) > 1 else float('nan')
    prauc  = average_precision_score(y_true, probs) if len(np.unique(y_true)) > 1 else float('nan')
    smgnn  = smgnn_auc.get(task, float('nan'))
    rows.append({
        'Task':          task,
        'ECFP+XGB AUC':  round(auc,   4),
        'SMILESGNN AUC': round(smgnn,  4),
        'Δ (XGB − GNN)': round(auc - smgnn, 4),
        'PR-AUC':        round(prauc, 4),
    })

df = pd.DataFrame(rows).set_index('Task')

# Append mean row
means = df.mean()
df.loc['MEAN'] = means
df.loc['MEAN', 'Δ (XGB − GNN)'] = means['ECFP+XGB AUC'] - means['SMILESGNN AUC']

def color_delta(val):
    if pd.isna(val): return ''
    color = 'green' if val > 0 else ('red' if val < 0 else 'black')
    return f'color: {color}'

display(df.style
    .format('{:.4f}')
    .applymap(color_delta, subset=['Δ (XGB − GNN)'])
    .set_caption('Tox21 Test-Set AUC-ROC: ECFP4+XGBoost vs. SMILESGNN (scaffold split)')
)

## 4. Global SHAP — which substructures drive each assay?

SHAP summary (beeswarm) plots show the 20 most influential ECFP bits for each task.  
X-axis: SHAP value (→ right = toward toxic).  
Color: bit value (purple = bit set, yellow = bit absent).

In [ ]:
# Compute SHAP values for all tasks on the test set
# (only labeled compounds per task)
shap_cache = {}
for task in TASKS:
    if task not in explainers:
        continue
    mask = ~np.isnan(y_test[:, TASKS.index(task)])
    sv   = explainers[task].shap_values(X_test[mask])
    shap_cache[task] = (sv, X_test[mask])
    print(f'  {task:<20} SHAP computed on {mask.sum()} compounds')

In [ ]:
# Beeswarm plot for the 4 tasks with most labeled data
show_tasks = sorted(shap_cache, key=lambda t: shap_cache[t][0].shape[0], reverse=True)[:4]

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('SHAP Summary — Top 20 ECFP4 Bits per Assay', fontsize=13)

for ax, task in zip(axes.flat, show_tasks):
    sv, X_sub = shap_cache[task]
    plt.sca(ax)
    shap.summary_plot(
        sv, X_sub,
        max_display=20,
        plot_type='dot',
        show=False,
        plot_size=None,
    )
    ax.set_title(task, fontsize=11)
    ax.set_xlabel('SHAP value', fontsize=9)

plt.tight_layout()
plt.savefig(MODEL_DIR / 'shap_beeswarm_4tasks.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved → shap_beeswarm_4tasks.png')

## 5. Molecule-level SHAP — atom-highlighted visualizations

For each toxic compound in the test set, map SHAP bit values back to the atoms
that generated those bits.  
**Red** = atom neighborhood pushes toward *toxic*.  
**Blue** = atom neighborhood pushes toward *non-toxic*.

In [ ]:
def explain_molecule(smiles, task, n_top_bits=10):
    """
    Compute and display atom-level SHAP explanation for one molecule + task.
    Returns (shap_values, atom_weights, predicted_prob).
    """
    t = TASKS.index(task)
    mol = Chem.MolFromSmiles(smiles)
    if mol is None or task not in models:
        return None, None, None

    fp, _ = smiles_to_ecfp([smiles], radius=RADIUS, nbits=NBITS)
    prob   = models[task].predict_proba(fp)[0, 1]
    sv     = explainers[task].shap_values(fp)[0]   # shape (nbits,)

    atom_w = shap_bits_to_atom_weights(mol, sv, radius=RADIUS, nbits=NBITS)

    # Draw
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    fig.suptitle(f'{task}  |  P(toxic) = {prob:.3f}', fontsize=11)

    # Left: atom-highlighted SHAP map
    SimilarityMaps.GetSimilarityMapFromWeights(
        mol, list(atom_w), colorMap=cm.RdBu_r, alpha=0.5, ax=axes[0]
    )
    axes[0].set_title('Atom SHAP weights', fontsize=9)
    axes[0].axis('off')

    # Right: top-N positive bits as bar chart
    top_idx  = np.argsort(sv)[::-1][:n_top_bits]
    axes[1].barh(
        [f'bit {i}' for i in top_idx],
        sv[top_idx],
        color=['#d73027' if v > 0 else '#4575b4' for v in sv[top_idx]]
    )
    axes[1].axvline(0, color='black', linewidth=0.8)
    axes[1].set_xlabel('SHAP value')
    axes[1].set_title(f'Top {n_top_bits} bits', fontsize=9)
    axes[1].invert_yaxis()

    plt.tight_layout()
    plt.show()
    return sv, atom_w, prob

In [ ]:
# Pick the task with most labeled toxic compounds in test set for visualization
focus_task = 'SR-MMP'   # mitochondrial membrane potential — most labeled
t_idx = TASKS.index(focus_task)

mask   = ~np.isnan(y_test[:, t_idx])
toxic  = (y_test[mask, t_idx] == 1)
toxic_smiles = [test_smiles[i] for i in np.where(mask)[0][toxic]][:6]

print(f'{focus_task}: {toxic.sum()} toxic compounds in test set')
print(f'Showing first {len(toxic_smiles)}\n')

for smi in toxic_smiles[:3]:
    explain_molecule(smi, focus_task)

## 6. Top toxic substructure gallery

Decode the highest mean-SHAP ECFP bits into their RDKit substructure fragments.  
These are the molecular patterns most consistently associated with toxicity.

In [ ]:
from rdkit.Chem import rdMolDescriptors

def decode_top_bits(task, n_bits=12):
    """
    Find molecules that set each top-SHAP bit, then draw the atom
    environment (substructure) that generated that bit.
    """
    sv, X_sub = shap_cache[task]
    # Mean absolute SHAP per bit (over labeled test compounds)
    mean_shap  = sv.mean(axis=0)         # average signed SHAP per bit
    top_bits   = np.argsort(mean_shap)[::-1][:n_bits]   # most toxic bits

    t_idx = TASKS.index(task)
    mask  = ~np.isnan(y_test[:, t_idx])
    task_smiles = [test_smiles[i] for i in np.where(mask)[0]]

    envs, legends = [], []
    for bit in top_bits:
        found = False
        for smi in task_smiles:
            mol = Chem.MolFromSmiles(smi)
            if mol is None:
                continue
            bi = {}
            fp = AllChem.GetMorganFingerprintAsBitVect(
                mol, RADIUS, nBits=NBITS, bitInfo=bi
            )
            if bit in bi:
                atom_idx, radius = bi[bit][0]
                env = Chem.FindAtomEnvironmentOfRadiusN(mol, radius, atom_idx)
                amap = {}
                submol = Chem.PathToSubmol(mol, env, atomMap=amap)
                envs.append(submol)
                legends.append(f'bit {bit}\nSHAP={mean_shap[bit]:.3f}')
                found = True
                break
        if not found:
            envs.append(Chem.MolFromSmiles('C'))   # placeholder
            legends.append(f'bit {bit} (not found)')

    img = Draw.MolsToGridImage(
        envs, molsPerRow=4, subImgSize=(250, 200),
        legends=legends,
    )
    return img

print(f'Top toxic substructures for {focus_task}:')
img = decode_top_bits(focus_task, n_bits=12)
display(img)

In [ ]:
# Repeat for NR-AhR (best AUC task)
print('Top toxic substructures for NR-AhR:')
img2 = decode_top_bits('NR-AhR', n_bits=12)
display(img2)

## Summary

| Model | Mean AUC-ROC | Notes |
|---|---|---|
| ECFP4 + XGBoost | **0.7052** | Per-task models, SHAP interpretable |
| SMILESGNN (GATv2 + Transformer) | **0.7284** | End-to-end, post-hoc attribution |

**Key observations:**
- SMILESGNN has a ~2.3 pp AUC advantage overall, but ECFP+XGBoost wins on NR-AhR (0.827 vs 0.782).
- ECFP+XGBoost interpretability is **faithful by construction** — SHAP values on bit features have zero compensation problem.
- The atom SHAP maps directly highlight the chemical motifs driving each assay, without any post-hoc attribution approximation.
- The top-bit substructure gallery provides chemically meaningful insight (e.g., aromatic amines, halogens, electron-rich systems for AhR).

**Next step**: AttentiveFP — graph model with inherent atom attention that should improve AUC while retaining direct interpretability.
